# Architecture

What is the underlying architecture of Kafi Streams?

Here is a dependency diagram:

```mermaid
---
title: Kafi Streams dependency diagram
---
classDiagram
    Streams <|-- TopologyNode
    Streams <|-- Kafi

    TopologyNode <|-- pydbsp
    TopologyNode <|-- msgpack
    TopologyNode <|-- cloudpickle
```

The two central classes in Kafi Streams are `TopologyNode` and `Streams`.

## TopologyNode

The `TopologyNode` class is the fluent API on top of [*pydbsp*](https://github.com/brurucy/pydbsp) by Bruno Rucy, and is heavily inspired by the Kafka Streams DSL.

The class is completely abstracted away from Kafka. It does not know anything about Kafka. It just receives inputs, processes them relationally using pydbsp, and returns the outputs. This is why it can serve also as a test harness similar to the "TopologyTestDriver" in Kafka Streams.

These are the dependencies of the TopologyNode class, from bottom to top.

### pydbsp

The by far most important building block is pydbsp by Bruno Rucy. It is the heart of Kafi Streams. It is the actual stream processing engine.

### msgpack

In pydbsp, the fundamental data type is the *ZSet*. ZSets are implemented as dictionaries in pydbsp, where the keys are rows and the values are weights (=integers), e.g.:
```python
{"row_1": 1, "row_2": 0, "row_3": -1}
```

As Kafi Streams is typically used on top of Kafka where the payloads are encoded in JSON (and Kafi converts the JSONs into Python dictionaries automatically), I needed a fast way to serialize these dictionaries into a hashable form and deserialize them back to dictionaries.

This is the task of msgpack.

### cloudpickle

cloudpickle is used for serializing/deserializing the global state of the topology (technically, the state of the pydbsp `evaluator`) since the built-in Python pickler is unable to serialize it.

## Streams

The `Streams` subclass of `TopologyNode` adds support for Kafka.

### Kafi

Kafi (the "old" part) provides all the Kafka support for Kafi Streams. It continuously consumes source topics, pushes the data to pydbsp, gets the outputs and produces them to sink topics.

Kafi also provides chunking/dechunking support which is required for checkpointing - where the checkpoints can go either to real Kafka or, through Kafi's "Kafka emulation", also to disk, S3 or Azure Blob Storage.


## About the Relationship of TopologyNode and Streams

What is the relationship between the two main classes of Kafi Streams, `TopologyNode` and `Streams`?

The [Quickstart](quickstart.ipynb) was based on the `Streams` subclass of the `TopologyNode` class because the aim was to give you the full picture from the start.

In the examples in the following chapters, however, for simplicity and brevity, we will often just use the `TopologyNode` class that has no connection with Kafka whatsoever.

`Streams`, as already alluded to above, is just about adding support for Kafka to `TopologyNode`. The actual stream processing in Kafi Streams is completely independent of Kafka - it could, in principle, be fed by any source and emit the output to any sink.

Here is a practical example - the code from the example in [Quickstart](quickstart.ipynb), but based on `TopologyNode` instead of `Streams`:

In [3]:
# 1. Boilerplate

import sys
sys.path.insert(1, "../..")

import importlib
import kafi.streams.topologynode
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

import logging
logging.basicConfig(level=logging.INFO)

# 2. Specify the Topology

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "joined"

## a) Clicks

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
    .filter(lambda r: r["view_time"] > 20)
    .distinct()
)

## b) Customers

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

## c) Join and Sink

sink_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "name": r_r["name"]}})
    .sink(sink_str)
)

# 3. Build the Topology

built_tn = Tn.build(sink_tn)


As you can see, the topology is defined identically. The only differences are:
* The connection to Kafka is left out, also in the sources and the sink specifications.
* The sources are specified using `Tn.source()` instead of `Streams.source()`.
* The build step is done using `Tn.build()` instead of `Streams.build()`.

Now how can supply data to the sources without Kafka? And how can we get the outputs of the processing?

To this end, we first repeat the code for the data generators:

In [4]:
import random, time

from faker import Faker

customers_int = 100

class ClickGenerator:
    def __init__(self):
        self.ts_int = int(time.time() * 1000)
        self.ts_step_int = 100
        self.customer_id_int = 0

    def generate(self):
        message_dict = {
            "key": None,
            "value": {"customer_id": random.randint(0, customers_int - 1),
                      "view_time": random.randint(10, 120),
                      "ts": self.ts_int},
        }
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict

class CustomerGenerator:
    def __init__(self):
        self.customer_id_int = 0
        self.customer_id_int_name_str_dict = {}
        fake = Faker()
        for customer_id_int in range(customers_int):
            name_str = fake.name()
            self.customer_id_int_name_str_dict[customer_id_int] = name_str

    def generate(self):
        customer_id_int = random.randint(0, customers_int - 1)
        message_dict = {
            "key": str(customer_id_int),
            "value": {"id": customer_id_int,
                      "name": self.customer_id_int_name_str_dict[customer_id_int]}
        }
        #
        return message_dict
    
click_generator = ClickGenerator()
for _ in range(3):
    print(click_generator.generate())

customer_generator = CustomerGenerator()
for _ in range(3):
    print(customer_generator.generate())


{'key': None, 'value': {'customer_id': 24, 'view_time': 68, 'ts': 1786537438152}}
{'key': None, 'value': {'customer_id': 21, 'view_time': 17, 'ts': 1786537438252}}
{'key': None, 'value': {'customer_id': 69, 'view_time': 79, 'ts': 1786537438352}}
{'key': '65', 'value': {'id': 65, 'name': 'Michael Robinson'}}
{'key': '13', 'value': {'id': 13, 'name': 'Danielle Kennedy'}}
{'key': '53', 'value': {'id': 53, 'name': 'Angela Conley'}}


And next, here is how that data can be fed into the built `TopologyNode` object directly:

In [ ]:
sink_m_dict_list = []
for i in range(100):
    # 1. Generate new data.

    click_m_dict_list = [click_generator.generate() for _ in range(0, 100)]
    customer_m_dict_list = [customer_generator.generate() for _ in range(0, 100)]

    # 2. Push the new data to the topology.

    built_tn.push({click_source_str: click_m_dict_list, customer_source_str: customer_m_dict_list})

    # 3. Incrementally process the new data and get the changes that it has triggered.

    sink_str_m_list_dict = built_tn.latest()
    m_list = sink_str_m_list_dict.get(sink_str, [])
    sink_m_dict_list += m_list

print(len(sink_m_dict_list))
print(sink_m_dict_list[-1])

89
{'value': {'customer_id': 2, 'view_time': 21, 'name': 'Virginia Horn'}}


In this loop, we do the following:
1. We generate new data.
2. We push the new data to the topology using the `push()` method of the `TopologyNode` class.
3. We incrementally process the new data and get the changes that it has triggered using the `latest()` method.

That is exactly what the `Streams` subclass does, just with Kafka for sources and sinks.
